In [ ]:
from datasets import load_dataset, load_from_disk
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
from functools import reduce
import ast
import numpy as np
from typing import List, Dict, Tuple
import json
import hashlib
import pandas as pd

BASE_PATH = "../../data/qa_datasets/v3/"

SAVE_DATASET_PATH = f"{BASE_PATH}/hotpotqa_distractor_validation/original"
SAVE_CONTEXTS_PATH = f"{BASE_PATH}/hotpotqa_distractor_validation/relevant_contexts.csv"
SAVE_TMP_CONTEXTS_PATH = f"{BASE_PATH}/hotpotqa_distractor_validation/tmp_hotpotqa_cntx.csv"
SAVE_QA_PATH = f"{BASE_PATH}/hotpotqa_distractor_validation/qa_pairs.csv"
SAVE_STATS_PATH = f"{BASE_PATH}/hotpotqa_distractor_validation/stats.json"
CNTX_MAX_LEN = 1024
CNTX_MIN_LEN = 64
MAX_QA_PAIRS = 2000

#### Скачиваем датасет с Hugging Face

In [ ]:
# https://huggingface.co/datasets/hotpotqa/hotpot_qa
valid_dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation") # 7405
valid_dataset.save_to_disk(SAVE_DATASET_PATH)

In [ ]:
valid_dataset = load_from_disk(SAVE_DATASET_PATH)

#### Извлекаем все уникальные контексты (разбиваем их на чанки, т.к. они достаточно длинные)

In [ ]:
base_supporting_facts = valid_dataset['supporting_facts']
base_contexts = valid_dataset['context']

In [ ]:
unique_contexts = dict()
contexts_qa_ids = defaultdict(set)

process = tqdm(range(valid_dataset.num_rows))
cntx_counter = 0
for i in process:
    supporting_facts = base_supporting_facts[i]
    contexts = base_contexts[i]
    qa_idx = i

    process.set_postfix_str(f"unique contexts: {len(unique_contexts)} / {cntx_counter}")

    for title in list(set(supporting_facts['title'])):
        title_index = contexts['title'].index(title)
        if len(title) < 1:
            raise ValueError
        rel_context = ' '.join(contexts['sentences'][title_index])
        cntx_hash = hashlib.sha1(rel_context.encode()).hexdigest()
        
        unique_contexts[cntx_hash] = (title, rel_context)
        contexts_qa_ids[cntx_hash].add(qa_idx)
        cntx_counter += 1

Preparing contexts list

In [ ]:
cntx_df_dict = {'context': [], 'title': [],'qa_ids': [], 'cntx_idx': []}
counter = 0
for cntx_hash in unique_contexts.keys():
    cntx_df_dict['title'].append(unique_contexts[cntx_hash][0])
    cntx_df_dict['context'].append(unique_contexts[cntx_hash][1])
    cntx_df_dict['qa_ids'].append(list(contexts_qa_ids[cntx_hash]))
    
    cntx_idx = counter
    cntx_df_dict['cntx_idx'].append(cntx_idx)
    counter += 1

In [ ]:
cntx_lens = list(map(lambda cntx: len(cntx), cntx_df_dict['context']))
plt.hist(cntx_lens, bins=50, density=True)
plt.title("Context len distribution")
plt.grid()
plt.show()

In [ ]:
cntx_df = pd.DataFrame(cntx_df_dict)
cntx_df.to_csv(SAVE_TMP_CONTEXTS_PATH, index=False)

In [ ]:
cntx_df

#### Фильтруем контексты по длине и максимальному количеству qa-пар

In [ ]:
cntx_df = pd.read_csv(SAVE_TMP_CONTEXTS_PATH)
cntx_df['qa_ids'] = cntx_df['qa_ids'].map(lambda v: ast.literal_eval(v))

In [ ]:
print(f"before: {cntx_df.shape}")

wolcnt_cntx_df = cntx_df[cntx_df['context'].str.len() <= CNTX_MAX_LEN].reset_index(drop=True)
print(f"after long contexts deletion: {wolcnt_cntx_df.shape}")

#
qa_idx_to_delete = set(reduce(lambda acc, v: acc + v, cntx_df[cntx_df['context'].str.len() > CNTX_MAX_LEN]['qa_ids'].tolist(), []))
wolcnt_cntx_df['qa_ids'] = [
    list(set(wolcnt_cntx_df['qa_ids'][i]).difference(qa_idx_to_delete)) 
    for i in tqdm(range(wolcnt_cntx_df.shape[0]))
]
wolcnt_cntx_df = wolcnt_cntx_df[wolcnt_cntx_df['qa_ids'].str.len() > 0].reset_index(drop=True)
print(f"after incomplete contexts deletion: {wolcnt_cntx_df.shape}")


wolscnt_cntx_df = wolcnt_cntx_df[wolcnt_cntx_df['context'].str.len() >= CNTX_MIN_LEN].reset_index(drop=True)
print(f"after small contexts deletion: {wolcnt_cntx_df.shape}")

#
qa_idx_to_delete = set(reduce(lambda acc, v: acc + v, wolcnt_cntx_df[wolcnt_cntx_df['context'].str.len() < CNTX_MIN_LEN]['qa_ids'].tolist(), []))
wolscnt_cntx_df['qa_ids'] = [
    list(set(wolscnt_cntx_df['qa_ids'][i]).difference(qa_idx_to_delete)) 
    for i in tqdm(range(wolscnt_cntx_df.shape[0]))]
wolscnt_cntx_df = wolscnt_cntx_df[wolscnt_cntx_df['qa_ids'].str.len() > 0].reset_index(drop=True)
print(f"after incomplete contexts deletion: {wolscnt_cntx_df.shape}")

#
unique_qa_ids = sorted(list(set(reduce(lambda acc, v: acc + v, wolscnt_cntx_df['qa_ids'].tolist(), []))))[:MAX_QA_PAIRS]
wolscnt_cntx_df['qa_ids'] = [
    list(set(wolscnt_cntx_df['qa_ids'][i]).intersection(unique_qa_ids))
    for i in range(wolscnt_cntx_df.shape[0])]
filtered_cntx_df = wolscnt_cntx_df[wolscnt_cntx_df['qa_ids'].str.len() > 0].reset_index(drop=True)
print(f"after ommiting qa-pairs: {filtered_cntx_df.shape}")

In [ ]:
cntx_lens = list(map(lambda cntx: len(cntx), filtered_cntx_df['context']))
plt.hist(cntx_lens, bins=50, density=True)
plt.title("Context len distribution")
plt.grid()
plt.show()

In [ ]:
filtered_cntx_df.to_csv(SAVE_CONTEXTS_PATH, index=False)

In [ ]:
filtered_cntx_df

#### Формируем отдельный датасет с qa-парами (с учётом имеющегося набора отфильтрованных контекстов)

In [ ]:
filtered_cntx_df = pd.read_csv(SAVE_CONTEXTS_PATH)
filtered_cntx_df['qa_ids'] = filtered_cntx_df['qa_ids'].map(lambda v: ast.literal_eval(v))

In [ ]:
filtered_cntx_df

In [ ]:
qa_df_dict = {'answer': [], 'question': [], 'qa_idx': [], 'cntx_ids': []}

all_questions = valid_dataset['question']
all_answers = valid_dataset['answer']

for qa_idx in tqdm(list(unique_qa_ids)):
    qa_df_dict['answer'].append(all_answers[qa_idx])
    qa_df_dict['question'].append(all_questions[qa_idx])
    qa_df_dict['qa_idx'].append(qa_idx)

qa_df_dict['cntx_ids'] = [[] for _ in range(len(qa_df_dict['question']))]
cntx_ids = filtered_cntx_df['cntx_idx'].tolist()
qa_ids_groups = filtered_cntx_df['qa_ids'].tolist()
noizy_contexts_counter = 0
for i in range(filtered_cntx_df.shape[0]):
    for qa_idx in qa_ids_groups[i]:
        try:
            cur_idx = qa_df_dict['qa_idx'].index(qa_idx)
        except ValueError:
            noizy_contexts_counter += 1
        else:
            qa_df_dict['cntx_ids'][cur_idx].append(cntx_ids[i])

In [ ]:
print(noizy_contexts_counter)

In [ ]:
answer_lens = list(map(lambda cntx: len(cntx), qa_df_dict['answer']))
plt.hist(answer_lens, bins=50)
plt.title("Answer len distribution")
plt.grid()
plt.show()

In [ ]:
question_lens = list(map(lambda cntx: len(cntx), qa_df_dict['question']))
plt.hist(question_lens, bins=50)
plt.title("Question len distribution")
plt.grid()
plt.show()

In [ ]:
qa_df = pd.DataFrame(qa_df_dict)
print(qa_df.shape)

In [ ]:
print("Количество уникальных вопросов: ", len(qa_df['question'].unique()))

In [ ]:
qa_df.to_csv(SAVE_QA_PATH, index=False)

In [ ]:
qa_df

#### Сохраняем статистику по полученным csv-файлам с контекстами и соответствующими qa-парами

In [ ]:
qa_df = pd.read_csv(SAVE_QA_PATH)
qa_df['cntx_ids'] = qa_df['cntx_ids'].map(lambda v: ast.literal_eval(v))
filtered_cntx_df = pd.read_csv(SAVE_CONTEXTS_PATH)
filtered_cntx_df['qa_ids'] = filtered_cntx_df['qa_ids'].map(lambda v: ast.literal_eval(v))

In [ ]:
def compute_stats(numbers: List[int]) -> Dict[str, float]:
    return {'median': float(round(np.median(numbers),5)),
            'mean': float(round(np.mean(numbers),5)),
            'std': float(round(np.std(numbers),5)),
            'min': float(round(min(numbers),5)),
            'max': float(round(max(numbers),5))}

In [ ]:
filtered_ds_statistics = {
    'contexts_amount': filtered_cntx_df.shape[0],
    'contexts_len': compute_stats(list(map(lambda v: len(v), filtered_cntx_df['context'].tolist()))),
    'qa_pairs_amount': qa_df.shape[0],
    'answers_len':  compute_stats(list(map(lambda v: len(v), qa_df['answer'].tolist()))),
    'questions_len': compute_stats(list(map(lambda v: len(v), qa_df['question'].tolist()))),
    'relevant_contexts_per_qa': compute_stats(list(map(lambda v: len(v), qa_df['cntx_ids'].tolist())))
}

In [ ]:
filtered_ds_statistics

In [ ]:
with open(SAVE_STATS_PATH, 'w', encoding='utf-8') as fd:
    json.dump(filtered_ds_statistics, fd, indent=1, ensure_ascii=False)